# Applying `MCube` to the 10x Visium HD CRC dataset

In [4]:
set.seed(20250502)

library(Matrix)
library(ggplot2)

library(spacexr)
library(MCube)

In [5]:
RAW_DATA_PATH <- "/import/home/share/zw/data/CRC"
DATA_PATH <- "/import/home/share/zw/pql/data/CRC"
RESULT_PATH <- "/import/home/share/zw/pql/results/CRC"

if (!dir.exists(file.path(RESULT_PATH, "VisiumHD"))) {
    dir.create(file.path(RESULT_PATH, "VisiumHD"), recursive = TRUE)
}

## Cell type deconvolution using `RCTD`

In [ ]:
# library(Seurat)

# FlexRef <- Read10X_h5(file.path(
#     RAW_DATA_PATH, "sc", "HumanColonCancer_Flex_Multiplex_count_filtered_feature_bc_matrix.h5"
# ))
# # MetaData <- readRDS(file.path(
# #     RAW_DATA_PATH, "sc", "FlexSeuratV5_MetaData.rds"
# # )) # See FlexSingleCell.R if not generated.

# meta <- read.csv(file.path(
#     RAW_DATA_PATH, "HumanColonCancer_VisiumHD/MetaData/SingleCell_MetaData.csv.gz"
# ))

# KpIdents <- names(which(table(meta$Level2) > 25))
# meta <- meta[meta$Level2 %in% KpIdents, ]
# FlexRef <- FlexRef[, meta$Barcode]

# CTRef <- meta$Level2
# CTRef <- gsub("/", "_", CTRef)
# CTRef <- as.factor(CTRef)
# names(CTRef) <- meta$Barcode

# reference <- Reference(FlexRef, CTRef, colSums(FlexRef))

In [ ]:
# library(Seurat)
# library(arrow)

# counts <- Read10X_h5(file.path(
#     RAW_DATA_PATH, "visium_hd",
#     "binned_outputs/square_016um/filtered_feature_bc_matrix.h5"
# ))

# coords <- read_parquet(
#     file.path(
#         RAW_DATA_PATH, "visium_hd",
#         "binned_outputs/square_016um/spatial/tissue_positions.parquet"
#     )
# )
# coords <- as.data.frame(coords)
# rownames(coords) <- coords$barcode
# coords <- coords[colnames(counts), ]
# coords <- coords[, c(6, 5)]

# nUMI <- colSums(counts)

# puck <- SpatialRNA(coords, counts, nUMI)
# barcodes <- colnames(puck@counts)

# myRCTD <- create.RCTD(puck, reference, max_cores = 32)
# myRCTD <- run.RCTD(myRCTD, doublet_mode = "doublet")

# saveRDS(
#     myRCTD,
#     file = file.path(
#         RESULT_PATH, "VisiumHD", "myRCTD_16um.rds"
#     )
# )

## Cell-type-specific SVG identification using `MCube`

Due to the high sparsity and significant noise in the Visium HD data, for the cell types of interest, we select bins that are confirmed to contain those specific cell types based on the results from `RCTD` (doublet mode) for further analysis.

In [6]:
myRCTD <- readRDS(file.path(RESULT_PATH, "VisiumHD", "myRCTD_16um.rds"))
weights_RCTD <- as.matrix(myRCTD@results$weights)
proportions_RCTD <- weights_RCTD / rowSums(weights_RCTD)
spot_effects_RCTD <- log(rowSums(weights_RCTD))
names(spot_effects_RCTD) <- rownames(weights_RCTD)
doublet_results_RCTD <- myRCTD@results$results_df

In [ ]:
gene_xenium <- colnames(readr::read_csv(
    file.path(DATA_PATH, "Xenium", "xenium_p2_counts.csv")
))[-1]

In [ ]:
sample_size_max <- 10000
celltype_threshold <- 100
for (celltype in colnames(proportions_RCTD)) {
    spots_used <- rownames(doublet_results_RCTD)[
        ((doublet_results_RCTD$spot_class == "singlet" |
            doublet_results_RCTD$spot_class == "doublet_uncertain") &
            doublet_results_RCTD$first_type == celltype
        ) |
            (doublet_results_RCTD$spot_class == "doublet_certain" &
                (doublet_results_RCTD$first_type == celltype |
                    doublet_results_RCTD$second_type == celltype))
    ]

    if (length(spots_used) > 0 & sum(proportions_RCTD[spots_used, celltype]) > celltype_threshold) {
        if (length(spots_used) > sample_size_max) {
            spots_used <- sample(spots_used, size = sample_size_max, replace = FALSE)
        }

        mcube_object <- createMCube(
            counts = t(as.matrix(myRCTD@originalSpatialRNA@counts[, spots_used])),
            coordinates = as.matrix(myRCTD@spatialRNA@coords[spots_used, ]),
            proportions = proportions_RCTD[spots_used, ],
            library_sizes = myRCTD@spatialRNA@nUMI[spots_used],
            reference = t(myRCTD@cell_type_info$info[[1]]),
            used_for_deconvolution = rownames(myRCTD@spatialRNA@counts),
            spot_effects = spot_effects_RCTD[spots_used],
            celltype_test = celltype, 
            # gene_test = gene_xenium,
            proportion_threshold = 0.01
        )
        mcube_object <- mcubeFitNull(
            mcube_object,
            num_workers = 70, num_threads = 1
        )
        mcube_object <- mcubeTest(
            mcube_object,
            num_workers = 70, num_threads = 1, shared_memory = TRUE
        )

        saveRDS(
            mcube_object,
            file = file.path(
                RESULT_PATH, "VisiumHD",
                paste0("mcube_", celltype, ".rds")
            )
        )
    }
}